In [45]:
from pathlib import Path
import json
import random
from collections import Counter, defaultdict

import pandas as pd
from datasets import get_dataset_config_names, load_dataset

REPO_ROOT = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine")
OUT_DIR = REPO_ROOT / "multilingual/data/processed/mmlu_prox"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HF_DATASET = "li-lab/MMLU-ProX"
DATASET_NAME = "mmlu_prox"

LANG_CONFIGS = {
    "russian": "ru",
    "chinese": "zh",
    "hindi": "hi",
    "italian": "it",
    "spanish": "es",
}

LETTERS = list("ABCDEFGHIJ")
EVAL_FRACTION = 0.15
SEED = 552

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


random.seed(SEED)

print("Output:", OUT_DIR)

Output: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mmlu_prox


In [3]:
configs = get_dataset_config_names(HF_DATASET)

print("Available configs:")
print(configs)

missing = [cfg for cfg in LANG_CONFIGS.values() if cfg not in configs]
assert not missing, f"Missing configs: {missing}"

print("\nSelected configs:")
for lang, cfg in LANG_CONFIGS.items():
    print(f"{lang:8s} -> {cfg}")

Available configs:
['af', 'ar', 'bn', 'cs', 'de', 'en', 'es', 'fr', 'hi', 'hu', 'id', 'it', 'ja', 'ko', 'mr', 'ne', 'pt', 'ru', 'sr', 'sw', 'te', 'th', 'uk', 'ur', 'vi', 'wo', 'yo', 'zh', 'zu']

Selected configs:
russian  -> ru
chinese  -> zh
hindi    -> hi
italian  -> it
spanish  -> es


In [4]:
for lang, cfg in LANG_CONFIGS.items():
    print("=" * 100)
    print(lang, cfg)

    ds = load_dataset(HF_DATASET, cfg)
    print(ds)

    for split in ds:
        print("split:", split)
        print("n:", len(ds[split]))
        print("columns:", ds[split].column_names)
        print("example:")
        print(json.dumps(dict(ds[split][0]), ensure_ascii=False, indent=2)[:10])
        break

russian ru
DatasetDict({
    validation: Dataset({
        features: ['question_id', 'question', 'option_0', 'option_1', 'option_2', 'option_3', 'option_4', 'option_5', 'option_6', 'option_7', 'option_8', 'option_9', 'answer', 'answer_index', 'cot_content', 'category', 'src', 'question_id_src'],
        num_rows: 70
    })
    test: Dataset({
        features: ['question_id', 'question', 'option_0', 'option_1', 'option_2', 'option_3', 'option_4', 'option_5', 'option_6', 'option_7', 'option_8', 'option_9', 'answer', 'answer_index', 'cot_content', 'category', 'src', 'question_id_src'],
        num_rows: 11759
    })
})
split: validation
n: 70
columns: ['question_id', 'question', 'option_0', 'option_1', 'option_2', 'option_3', 'option_4', 'option_5', 'option_6', 'option_7', 'option_8', 'option_9', 'answer', 'answer_index', 'cot_content', 'category', 'src', 'question_id_src']
example:
{
  "quest
chinese zh
DatasetDict({
    validation: Dataset({
        features: ['question_id', 'question'

# helpers

In [16]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records



def normalize_options(row):
    options = []

    for i in range(10):
        key = f"option_{i}"
        value = row.get(key, None)

        if value is None:
            continue

        value = str(value).strip()
        if not value or value.lower() == "none" or value.lower() == "null":
            continue

        options.append({
            "original_index": i,
            "text": value,
        })

    if len(options) < 3:
        return None

    return options


def normalize_answer(row, options):
    answer_index = row.get("answer_index", None)

    if isinstance(answer_index, int):
        idx = answer_index
    elif str(answer_index).isdigit():
        idx = int(answer_index)
    else:
        return None

    original_indices = [opt["original_index"] for opt in options]

    if idx not in original_indices:
        return None

    new_index = original_indices.index(idx)

    if new_index >= len(LETTERS):
        return None

    return LETTERS[new_index]


def format_prompt(question, options):
    options_text = "\n".join(
        f"{LETTERS[i]}) {opt['text']}"
        for i, opt in enumerate(options)
    )
    return f"{question.strip()}\n\n{options_text}"


def make_source_id(language, row, hf_split):
    if row.get("question_id_src", None) not in (None, ""):
        return f"mmlu_prox_{language}_{hf_split}_{row['question_id_src']}"
    if row.get("question_id", None) not in (None, ""):
        return f"mmlu_prox_{language}_{hf_split}_{row['question_id']}"

    raw = f"{language}\n{hf_split}\n{row.get('question', '')}\n{row.get('answer', '')}"
    import hashlib
    h = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]
    return f"mmlu_prox_{language}_{hf_split}_{h}"

def normalize_record(row, language, hf_split):
    row = dict(row)

    question = str(row.get("question", "")).strip()
    if not question:
        return None

    options = normalize_options(row)
    if options is None:
        return None

    answer = normalize_answer(row, options)
    if answer not in LETTERS[:len(options)]:
        return None

    source_id = make_source_id(language, row, hf_split)
    subject = str(row.get("category", "unknown")).strip() or "unknown"

    return {
        "source_id": source_id,
        "language": language,
        "dataset": DATASET_NAME,
        "prompt": format_prompt(question, options),
        "answer": answer,
        "id": source_id,
        "split": "all",
        "task_type": "mcq",
        "metadata": {
            "source_dataset": HF_DATASET,
            "hf_config": LANG_CONFIGS[language],
            "hf_split": hf_split,
            "question_id": row.get("question_id"),
            "question_id_src": row.get("question_id_src"),
            "subject": subject,
            "category": subject,
            "src": row.get("src"),
            "answer_index": row.get("answer_index"),
            "n_options": len(options),
            "option_letters": LETTERS[:len(options)],
            "original_option_indices": [opt["original_index"] for opt in options],
        },
    }

def dedupe_by_source_id(records):
    seen = set()
    out = []

    for r in records:
        sid = r["source_id"]
        if sid in seen:
            continue
        seen.add(sid)
        out.append(r)

    return out

# load

In [17]:
records_by_language = {}
rejected_by_language = {}

for language, cfg in LANG_CONFIGS.items():
    print("=" * 100)
    print("Loading:", language, cfg)

    ds = load_dataset(HF_DATASET, cfg)

    records = []
    rejected = 0

    for hf_split in ds:
        for row in ds[hf_split]:
            rec = normalize_record(row, language, hf_split)
            if rec is None:
                rejected += 1
                continue
            records.append(rec)

    records = dedupe_by_source_id(records)

    records_by_language[language] = records
    rejected_by_language[language] = rejected

    print("valid:", len(records))
    print("rejected:", rejected)

    assert len(records) > 0

Loading: russian ru
valid: 11829
rejected: 0
Loading: chinese zh
valid: 11829
rejected: 0
Loading: hindi hi
valid: 11829
rejected: 0
Loading: italian it
valid: 11829
rejected: 0
Loading: spanish es
valid: 11829
rejected: 0


In [22]:
summary_rows = []

for language, records in records_by_language.items():
    subjects = Counter(r["metadata"]["subject"] for r in records)
    answers = Counter(r["answer"] for r in records)
    n_options = Counter(r["metadata"]["n_options"] for r in records)

    row = {
        "language": language,
        "n_records": len(records),
        "n_subjects": len(subjects),
        "n_rejected": rejected_by_language[language],
    }

    for letter in LETTERS:
        row[f"answer_{letter}"] = answers.get(letter, 0)

    for k in range(3, 11):
        row[f"n_options_{k}"] = n_options.get(k, 0)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

,language,n_records,n_subjects,n_rejected,answer_A,answer_B,answer_C,answer_D,answer_E,answer_F,answer_G,answer_H,answer_I,answer_J,n_options_3,n_options_4,n_options_5,n_options_6,n_options_7,n_options_8,n_options_9,n_options_10
0,russian,11829,14,0,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
1,chinese,11829,14,0,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,773,9895
2,hindi,11829,14,0,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
3,italian,11829,14,0,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
4,spanish,11829,14,0,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896


In [23]:
for language, records in records_by_language.items():
    print("=" * 100)
    print(language)
    print("Total:", len(records))
    print("Rejected:", rejected_by_language[language])
    print("Answer distribution:", dict(sorted(Counter(r["answer"] for r in records).items())))
    print("Option-count distribution:", dict(sorted(Counter(r["metadata"]["n_options"] for r in records).items())))
    print("Subjects:", len(set(r["metadata"]["subject"] for r in records)))
    print("Top subjects:")
    for subject, n in Counter(r["metadata"]["subject"] for r in records).most_common(15):
        print(f"  {subject}: {n}")

russian
Total: 11829
Rejected: 0
Answer distribution: {'A': 1369, 'B': 1319, 'C': 1292, 'D': 1314, 'E': 1133, 'F': 1114, 'G': 1157, 'H': 1104, 'I': 1090, 'J': 937}
Option-count distribution: {3: 19, 4: 545, 5: 51, 6: 88, 7: 152, 8: 306, 9: 772, 10: 9896}
Subjects: 14
Top subjects:
  math: 1356
  physics: 1304
  chemistry: 1137
  engineering: 974
  law: 964
  other: 929
  economics: 849
  psychology: 803
  business: 794
  biology: 722
  health: 692
  philosophy: 504
  computer science: 415
  history: 386
chinese
Total: 11829
Rejected: 0
Answer distribution: {'A': 1369, 'B': 1319, 'C': 1292, 'D': 1314, 'E': 1133, 'F': 1114, 'G': 1157, 'H': 1104, 'I': 1090, 'J': 937}
Option-count distribution: {3: 19, 4: 545, 5: 51, 6: 88, 7: 152, 8: 306, 9: 773, 10: 9895}
Subjects: 14
Top subjects:
  math: 1356
  physics: 1304
  chemistry: 1137
  engineering: 974
  law: 964
  other: 929
  economics: 849
  psychology: 803
  business: 794
  biology: 722
  health: 692
  philosophy: 504
  computer science: 4

In [24]:
def stratified_train_eval_split(records, eval_fraction=0.15, seed=552):
    rng = random.Random(seed)

    groups = defaultdict(list)
    for r in records:
        key = (
            r["metadata"].get("subject", "unknown"),
            r["answer"],
            r["metadata"].get("n_options", 10),
        )
        groups[key].append(r)

    train_records = []
    eval_records = []

    for group in groups.values():
        group = list(group)
        rng.shuffle(group)

        if len(group) >= 10:
            n_eval = max(1, int(round(len(group) * eval_fraction)))
        elif len(group) >= 2:
            n_eval = 1
        else:
            n_eval = 0

        eval_part = group[:n_eval]
        train_part = group[n_eval:]

        eval_records.extend(eval_part)
        train_records.extend(train_part)

    rng.shuffle(train_records)
    rng.shuffle(eval_records)

    for r in train_records:
        r["split"] = "train"
        r["metadata"]["local_split"] = "train"

    for r in eval_records:
        r["split"] = "eval"
        r["metadata"]["local_split"] = "eval"

    return train_records, eval_records


train_by_language = {}
eval_by_language = {}

for language, records in records_by_language.items():
    train_records, eval_records = stratified_train_eval_split(
        records,
        eval_fraction=EVAL_FRACTION,
        seed=SEED,
    )

    train_by_language[language] = train_records
    eval_by_language[language] = eval_records

    print(f"{language:8s} total={len(records):5d} train={len(train_records):5d} eval={len(eval_records):5d}")

russian  total=11829 train= 9968 eval= 1861
chinese  total=11829 train= 9967 eval= 1862
hindi    total=11829 train= 9968 eval= 1861
italian  total=11829 train= 9968 eval= 1861
spanish  total=11829 train= 9968 eval= 1861


In [25]:
for language in LANG_CONFIGS:
    train_ids = {r["source_id"] for r in train_by_language[language]}
    eval_ids = {r["source_id"] for r in eval_by_language[language]}

    overlap = train_ids & eval_ids
    print(language, "overlap:", len(overlap))
    assert not overlap

russian overlap: 0
chinese overlap: 0
hindi overlap: 0
italian overlap: 0
spanish overlap: 0


# split, no cross-language contamination

In [26]:
def get_base_id(record):
    qid_src = record["metadata"].get("question_id_src")
    if qid_src not in (None, ""):
        return f"question_id_src_{qid_src}"

    qid = record["metadata"].get("question_id")
    if qid not in (None, ""):
        return f"question_id_{qid}"

    return record["source_id"]


def grouped_train_eval_split(records_by_language, eval_fraction=0.15, seed=552):
    rng = random.Random(seed)

    groups = defaultdict(list)

    for language, records in records_by_language.items():
        for r in records:
            base_id = get_base_id(r)
            subject = r["metadata"].get("subject", "unknown")
            n_options = r["metadata"].get("n_options", 10)
            key = (base_id, subject, n_options)
            groups[key].append(r)

    # Stratify approximately by subject and n_options at the group level.
    strata = defaultdict(list)
    for (base_id, subject, n_options), group_records in groups.items():
        strata[(subject, n_options)].append((base_id, group_records))

    train_by_language = {lang: [] for lang in records_by_language}
    eval_by_language = {lang: [] for lang in records_by_language}

    for stratum_key, group_list in strata.items():
        group_list = list(group_list)
        rng.shuffle(group_list)

        if len(group_list) >= 10:
            n_eval = max(1, int(round(len(group_list) * eval_fraction)))
        elif len(group_list) >= 2:
            n_eval = 1
        else:
            n_eval = 0

        eval_groups = group_list[:n_eval]
        train_groups = group_list[n_eval:]

        for _, group_records in train_groups:
            for r in group_records:
                rr = dict(r)
                rr["metadata"] = dict(rr["metadata"])
                rr["split"] = "train"
                rr["metadata"]["local_split"] = "train"
                rr["metadata"]["base_question_id"] = get_base_id(r)
                train_by_language[rr["language"]].append(rr)

        for _, group_records in eval_groups:
            for r in group_records:
                rr = dict(r)
                rr["metadata"] = dict(rr["metadata"])
                rr["split"] = "eval"
                rr["metadata"]["local_split"] = "eval"
                rr["metadata"]["base_question_id"] = get_base_id(r)
                eval_by_language[rr["language"]].append(rr)

    for language in records_by_language:
        rng.shuffle(train_by_language[language])
        rng.shuffle(eval_by_language[language])

    return train_by_language, eval_by_language


train_by_language, eval_by_language = grouped_train_eval_split(
    records_by_language,
    eval_fraction=EVAL_FRACTION,
    seed=SEED,
)

for language in LANG_CONFIGS:
    print(
        f"{language:8s} "
        f"total={len(records_by_language[language]):5d} "
        f"train={len(train_by_language[language]):5d} "
        f"eval={len(eval_by_language[language]):5d}"
    )

russian  total=11829 train=10043 eval= 1786
chinese  total=11829 train=10043 eval= 1786
hindi    total=11829 train=10043 eval= 1786
italian  total=11829 train=10043 eval= 1786
spanish  total=11829 train=10043 eval= 1786


In [27]:
train_base_ids = set()
eval_base_ids = set()

for language in LANG_CONFIGS:
    for r in train_by_language[language]:
        train_base_ids.add(r["metadata"]["base_question_id"])
    for r in eval_by_language[language]:
        eval_base_ids.add(r["metadata"]["base_question_id"])

overlap = train_base_ids & eval_base_ids

print("train base ids:", len(train_base_ids))
print("eval base ids:", len(eval_base_ids))
print("base-id overlap:", len(overlap))

assert not overlap

train base ids: 10043
eval base ids: 1786
base-id overlap: 0


In [28]:
for language in LANG_CONFIGS:
    train_ids = {r["source_id"] for r in train_by_language[language]}
    eval_ids = {r["source_id"] for r in eval_by_language[language]}
    overlap = train_ids & eval_ids
    print(language, "source_id overlap:", len(overlap))
    assert not overlap

russian source_id overlap: 0
chinese source_id overlap: 0
hindi source_id overlap: 0
italian source_id overlap: 0
spanish source_id overlap: 0


# save

## full train/eval sets

In [ ]:
# save per-language files
for language in LANG_CONFIGS:
    all_path = OUT_DIR / f"mmlu_prox_{language}_all.jsonl"
    train_path = OUT_DIR / f"mmlu_prox_{language}_train.jsonl"
    eval_path = OUT_DIR / f"mmlu_prox_{language}_eval.jsonl"

    all_records_out = []
    for r in records_by_language[language]:
        rr = dict(r)
        rr["split"] = "all"
        rr["metadata"] = dict(rr["metadata"])
        rr["metadata"].pop("local_split", None)
        # keep base_question_id also in all file if already present
        if "base_question_id" not in rr["metadata"]:
            rr["metadata"]["base_question_id"] = get_base_id(rr)
        all_records_out.append(rr)

    write_jsonl(all_records_out, all_path)
    write_jsonl(train_by_language[language], train_path)
    write_jsonl(eval_by_language[language], eval_path)

    print(language)
    print(" ", all_path.relative_to(REPO_ROOT), len(all_records_out))
    print(" ", train_path.relative_to(REPO_ROOT), len(train_by_language[language]))
    print(" ", eval_path.relative_to(REPO_ROOT), len(eval_by_language[language]))

russian
  multilingual/data/processed/mmlu_prox/mmlu_prox_russian_all.jsonl 11829
  multilingual/data/processed/mmlu_prox/mmlu_prox_russian_train.jsonl 10043
  multilingual/data/processed/mmlu_prox/mmlu_prox_russian_eval.jsonl 1786
chinese
  multilingual/data/processed/mmlu_prox/mmlu_prox_chinese_all.jsonl 11829
  multilingual/data/processed/mmlu_prox/mmlu_prox_chinese_train.jsonl 10043
  multilingual/data/processed/mmlu_prox/mmlu_prox_chinese_eval.jsonl 1786
hindi
  multilingual/data/processed/mmlu_prox/mmlu_prox_hindi_all.jsonl 11829
  multilingual/data/processed/mmlu_prox/mmlu_prox_hindi_train.jsonl 10043
  multilingual/data/processed/mmlu_prox/mmlu_prox_hindi_eval.jsonl 1786
italian
  multilingual/data/processed/mmlu_prox/mmlu_prox_italian_all.jsonl 11829
  multilingual/data/processed/mmlu_prox/mmlu_prox_italian_train.jsonl 10043
  multilingual/data/processed/mmlu_prox/mmlu_prox_italian_eval.jsonl 1786
spanish
  multilingual/data/processed/mmlu_prox/mmlu_prox_spanish_all.jsonl 1182

In [30]:
# Save combined train/eval files across all five languages

all_train = []
all_eval = []

for language in LANG_CONFIGS:
    all_train.extend(train_by_language[language])
    all_eval.extend(eval_by_language[language])

random.Random(SEED).shuffle(all_train)
random.Random(SEED).shuffle(all_eval)

combined_train_path = OUT_DIR / "mmlu_prox_all_train.jsonl"
combined_eval_path = OUT_DIR / "mmlu_prox_all_eval.jsonl"

write_jsonl(all_train, combined_train_path)
write_jsonl(all_eval, combined_eval_path)

print(combined_train_path.relative_to(REPO_ROOT), len(all_train))
print(combined_eval_path.relative_to(REPO_ROOT), len(all_eval))

print("\nCombined train by language:")
print(pd.Series([r["language"] for r in all_train]).value_counts().sort_index())

print("\nCombined eval by language:")
print(pd.Series([r["language"] for r in all_eval]).value_counts().sort_index())

multilingual/data/processed/mmlu_prox/mmlu_prox_all_train.jsonl 50215
multilingual/data/processed/mmlu_prox/mmlu_prox_all_eval.jsonl 8930

Combined train by language:
chinese    10043
hindi      10043
italian    10043
russian    10043
spanish    10043
Name: count, dtype: int64

Combined eval by language:
chinese    1786
hindi      1786
italian    1786
russian    1786
spanish    1786
Name: count, dtype: int64


## 5k train subsets, stratified by more regional/humanities like subjects

In [71]:
# Save 5k train subsets.
# These are sampled only from the contamination-safe train split.
# The 5k files are mildly biased toward regional subjects.

SUBSET_DIR = OUT_DIR / "subsets"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_N_PER_LANGUAGE = 5000

SUBSET_SEEDS = {
    "russian": 5521,
    "chinese": 5522,
    "hindi": 5523,
    "italian": 5524,
    "spanish": 5525,
}

REGIONAL_SUBJECTS = {
    "law",
    "history",
    "economics",
    "business",
    "philosophy",
    "psychology",
    "other",
}

GENERAL_REASONING_SUBJECTS = {
    "math",
    "physics",
    "chemistry",
    "biology",
    "engineering",
    "computer science",
}

HEALTH_SUBJECTS = {
    "health",
}

SUBJECT_WEIGHTS = {
    # Regional / cultural / society-oriented subjects
    "law": 1.6,
    "history": 1.6,
    "economics": 1.6,
    "business": 1.6,
    "philosophy": 1.6,
    "psychology": 1.6,
    "other": 1.6,

    # General hard-reasoning subjects; keep them, but do not prioritize them
    "math": 1.0,
    "physics": 1.0,
    "chemistry": 1.0,
    "engineering": 1.0,
    "computer science": 1.0,

    # Relevant to medicine / health / applied knowledge
    "biology": 1.4,
    "health": 1.4,
}


def weighted_sample_without_replacement(records, n, seed):
    rng = random.Random(seed)

    remaining = list(records)
    selected = []

    while remaining and len(selected) < n:
        weights = [
            SUBJECT_WEIGHTS.get(
                r["metadata"].get("subject", "unknown"),
                1.0,
            )
            for r in remaining
        ]

        chosen = rng.choices(remaining, weights=weights, k=1)[0]
        selected.append(chosen)
        remaining.remove(chosen)

    rng.shuffle(selected)
    return selected

In [72]:
# per language
train_5k_by_language = {}

for language in LANG_CONFIGS:
    records = list(train_by_language[language])

    subset = weighted_sample_without_replacement(
        records,
        n=min(SUBSET_N_PER_LANGUAGE, len(records)),
        seed=SUBSET_SEEDS[language],
    )

    train_5k_by_language[language] = subset

    path = SUBSET_DIR / f"mmlu_prox_{language}_train_5k.jsonl"
    write_jsonl(subset, path)

    print(f"{language:8s} {len(subset):5d} -> {path.relative_to(REPO_ROOT)}")

russian   5000 -> multilingual/data/processed/mmlu_prox/subsets/mmlu_prox_russian_train_5k.jsonl
chinese   5000 -> multilingual/data/processed/mmlu_prox/subsets/mmlu_prox_chinese_train_5k.jsonl
hindi     5000 -> multilingual/data/processed/mmlu_prox/subsets/mmlu_prox_hindi_train_5k.jsonl
italian   5000 -> multilingual/data/processed/mmlu_prox/subsets/mmlu_prox_italian_train_5k.jsonl
spanish   5000 -> multilingual/data/processed/mmlu_prox/subsets/mmlu_prox_spanish_train_5k.jsonl


In [73]:
# combined 25k train subset
combined_train_5k = []

for language in LANG_CONFIGS:
    combined_train_5k.extend(train_5k_by_language[language])

random.Random(5520).shuffle(combined_train_5k)

combined_5k_path = SUBSET_DIR / "mmlu_prox_all_train_5k_per_language.jsonl"
write_jsonl(combined_train_5k, combined_5k_path)

print(combined_5k_path.relative_to(REPO_ROOT), len(combined_train_5k))

print("\nCombined 5k train by language:")
print(pd.Series([r["language"] for r in combined_train_5k]).value_counts().sort_index())

multilingual/data/processed/mmlu_prox/subsets/mmlu_prox_all_train_5k_per_language.jsonl 25000

Combined 5k train by language:
chinese    5000
hindi      5000
italian    5000
russian    5000
spanish    5000
Name: count, dtype: int64


In [74]:
# check no base-question contamination between any train subset and eval.
eval_base_ids = set()
for language in LANG_CONFIGS:
    for r in eval_by_language[language]:
        eval_base_ids.add(r["metadata"]["base_question_id"])

for language in LANG_CONFIGS:
    subset_base_ids = {r["metadata"]["base_question_id"] for r in train_5k_by_language[language]}
    overlap = subset_base_ids & eval_base_ids
    print(language, "5k train vs eval base-id overlap:", len(overlap))
    assert not overlap

combined_5k_base_ids = {r["metadata"]["base_question_id"] for r in combined_train_5k}
overlap = combined_5k_base_ids & eval_base_ids

print("combined 5k train vs eval base-id overlap:", len(overlap))
assert not overlap

russian 5k train vs eval base-id overlap: 0
chinese 5k train vs eval base-id overlap: 0
hindi 5k train vs eval base-id overlap: 0
italian 5k train vs eval base-id overlap: 0
spanish 5k train vs eval base-id overlap: 0
combined 5k train vs eval base-id overlap: 0


In [75]:
regional_subject_rows = []

for language, records in train_5k_by_language.items():
    counts = Counter(r["metadata"].get("subject", "unknown") for r in records)

    for subject, n in sorted(counts.items()):
        regional_subject_rows.append({
            "subset": "regional_weighted_5k",
            "language": language,
            "subject": subject,
            "n": n,
        })

regional_subject_df = pd.DataFrame(regional_subject_rows)
regional_subject_df

,subset,language,subject,n
0,regional_weighted_5k,russian,biology,316
1,regional_weighted_5k,russian,business,379
2,regional_weighted_5k,russian,chemistry,411
3,regional_weighted_5k,russian,computer science,148
4,regional_weighted_5k,russian,economics,421
5,regional_weighted_5k,russian,engineering,328
6,regional_weighted_5k,russian,health,306
7,regional_weighted_5k,russian,history,190
8,regional_weighted_5k,russian,law,486
9,regional_weighted_5k,russian,math,467


In [76]:
for path in sorted(OUT_DIR.glob("*.jsonl")):
    n = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"{n:7d}  {path.relative_to(REPO_ROOT)}")

print("\nSubsets:")
for path in sorted(SUBSET_DIR.glob("*.jsonl")):
    n = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"{n:7d}  {path.relative_to(REPO_ROOT)}")

   8930  multilingual/data/processed/mmlu_prox/mmlu_prox_all_eval.jsonl
  50215  multilingual/data/processed/mmlu_prox/mmlu_prox_all_train.jsonl
  11829  multilingual/data/processed/mmlu_prox/mmlu_prox_chinese_all.jsonl
   1786  multilingual/data/processed/mmlu_prox/mmlu_prox_chinese_eval.jsonl
  10043  multilingual/data/processed/mmlu_prox/mmlu_prox_chinese_train.jsonl
  11829  multilingual/data/processed/mmlu_prox/mmlu_prox_hindi_all.jsonl
   1786  multilingual/data/processed/mmlu_prox/mmlu_prox_hindi_eval.jsonl
  10043  multilingual/data/processed/mmlu_prox/mmlu_prox_hindi_train.jsonl
  11829  multilingual/data/processed/mmlu_prox/mmlu_prox_italian_all.jsonl
   1786  multilingual/data/processed/mmlu_prox/mmlu_prox_italian_eval.jsonl
  10043  multilingual/data/processed/mmlu_prox/mmlu_prox_italian_train.jsonl
  11829  multilingual/data/processed/mmlu_prox/mmlu_prox_russian_all.jsonl
   1786  multilingual/data/processed/mmlu_prox/mmlu_prox_russian_eval.jsonl
  10043  multilingual/data

# summary

In [77]:
from collections import Counter
import json
import pandas as pd

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def summarize_records(records, name):
    rows = []

    for language in sorted(set(r["language"] for r in records)):
        lang_records = [r for r in records if r["language"] == language]

        answers = Counter(r["answer"] for r in lang_records)
        subjects = Counter(r["metadata"].get("subject", "unknown") for r in lang_records)
        n_options = Counter(r["metadata"].get("n_options", None) for r in lang_records)

        row = {
            "name": name,
            "language": language,
            "n": len(lang_records),
            "n_subjects": len(subjects),
        }

        for letter in LETTERS:
            row[f"answer_{letter}"] = answers.get(letter, 0)

        for k in range(3, 11):
            row[f"n_options_{k}"] = n_options.get(k, 0)

        rows.append(row)

    return pd.DataFrame(rows)


all_summaries = []

for split_name in ["all", "train", "eval"]:
    for language in LANG_CONFIGS:
        path = OUT_DIR / f"mmlu_prox_{language}_{split_name}.jsonl"
        records = load_jsonl(path)
        all_summaries.append(summarize_records(records, split_name))

summary_df = pd.concat(all_summaries, ignore_index=True)
summary_df

,name,language,n,n_subjects,answer_A,answer_B,answer_C,answer_D,answer_E,answer_F,answer_G,answer_H,answer_I,answer_J,n_options_3,n_options_4,n_options_5,n_options_6,n_options_7,n_options_8,n_options_9,n_options_10
0,all,russian,11829,14,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
1,all,chinese,11829,14,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,773,9895
2,all,hindi,11829,14,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
3,all,italian,11829,14,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
4,all,spanish,11829,14,1369,1319,1292,1314,1133,1114,1157,1104,1090,937,19,545,51,88,152,306,772,9896
5,train,russian,10043,14,1157,1135,1099,1105,975,939,978,931,927,797,15,463,40,72,127,259,656,8411
6,train,chinese,10043,14,1157,1135,1099,1105,975,939,978,931,927,797,15,463,40,72,127,259,657,8410
7,train,hindi,10043,14,1157,1135,1099,1105,975,939,978,931,927,797,15,463,40,72,127,259,656,8411
8,train,italian,10043,14,1157,1135,1099,1105,975,939,978,931,927,797,15,463,40,72,127,259,656,8411
9,train,spanish,10043,14,1157,1135,1099,1105,975,939,978,931,927,797,15,463,40,72,127,259,656,8411


In [78]:
answer_rows = []

for split_name in ["all", "train", "eval"]:
    for language in LANG_CONFIGS:
        path = OUT_DIR / f"mmlu_prox_{language}_{split_name}.jsonl"
        records = load_jsonl(path)

        counts = Counter(r["answer"] for r in records)
        row = {
            "split": split_name,
            "language": language,
            "n": len(records),
        }

        for letter in LETTERS:
            row[letter] = counts.get(letter, 0)

        answer_rows.append(row)

answer_df = pd.DataFrame(answer_rows)
answer_df

,split,language,n,A,B,C,D,E,F,G,H,I,J
0,all,russian,11829,1369,1319,1292,1314,1133,1114,1157,1104,1090,937
1,all,chinese,11829,1369,1319,1292,1314,1133,1114,1157,1104,1090,937
2,all,hindi,11829,1369,1319,1292,1314,1133,1114,1157,1104,1090,937
3,all,italian,11829,1369,1319,1292,1314,1133,1114,1157,1104,1090,937
4,all,spanish,11829,1369,1319,1292,1314,1133,1114,1157,1104,1090,937
5,train,russian,10043,1157,1135,1099,1105,975,939,978,931,927,797
6,train,chinese,10043,1157,1135,1099,1105,975,939,978,931,927,797
7,train,hindi,10043,1157,1135,1099,1105,975,939,978,931,927,797
8,train,italian,10043,1157,1135,1099,1105,975,939,978,931,927,797
9,train,spanish,10043,1157,1135,1099,1105,975,939,978,931,927,797


In [79]:
subject_rows = []

for split_name in ["all", "train", "eval"]:
    for language in LANG_CONFIGS:
        path = OUT_DIR / f"mmlu_prox_{language}_{split_name}.jsonl"
        records = load_jsonl(path)

        counts = Counter(r["metadata"].get("subject", "unknown") for r in records)

        for subject, n in sorted(counts.items()):
            subject_rows.append({
                "split": split_name,
                "language": language,
                "subject": subject,
                "n": n,
            })

subject_df = pd.DataFrame(subject_rows)
subject_df

,split,language,subject,n
0,all,russian,biology,722
1,all,russian,business,794
2,all,russian,chemistry,1137
3,all,russian,computer science,415
4,all,russian,economics,849
5,all,russian,engineering,974
6,all,russian,health,692
7,all,russian,history,386
8,all,russian,law,964
9,all,russian,math,1356
